# Full candidate M0 audit — fresh Qwen3-1.7B

Before scaling M1, measure the untrained pinned 1.7B base on all 180 candidate units: 30 F +90 R +60 new relational units, seven questions each. Use only m1_dev and n_rel_train: 1,260 questions in each declared mode, **2,520 predictions, zero training updates, no adapter loaded and no final-test generation**.

Keep the current candidate unchanged. The static acceptance review found surface overlaps, including a real book-title collision for The Silver Citadel; those records are flagged, not silently renamed/filtered. Audit completion or zero matches do not accept the dataset or certify absent internal knowledge. Review actual matches/mentions, format/cap issues and collision context before full training.

Use the existing Kaggle GPU kernel. Publish this runner/config/review metadata first. HF_TOKEN is optional for public-model downloads; no new checkpoint or model upload is performed. Download the final evidence ZIP.


In [ ]:
import os, subprocess, sys, tempfile
from pathlib import Path

remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect to the Kaggle GPU kernel.'
repo_dir = Path(tempfile.mkdtemp(prefix='full-m0-qwen17-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('src/evaluation/full_m0.py').is_file(), 'Publish the full M0 runner/config/review files first.'
assert Path('configs/evaluation/m0_qwen17_full_v1.json').is_file()


In [ ]:
assert Path('/kaggle/working').is_dir()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m1-tiny.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-c', 'import torch, transformers; assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__)'], check=True)


In [ ]:
import json

plan_dir = Path(tempfile.mkdtemp(prefix='full-m0-qwen17-plan-', dir=remote_root))
subprocess.run([sys.executable, '-m', 'src.evaluation.full_m0', '--config', 'configs/evaluation/m0_qwen17_full_v1.json', '--output-dir', str(plan_dir), '--mode', 'plan'], check=True)
plan = json.loads((plan_dir / 'plan.json').read_text())
assert plan['planned_predictions'] == 2520 and plan['knowledge_units'] == 180
assert plan['training_updates'] == 0 and not plan['adapters_loaded'] and not plan['dataset_accepted']
print('Non-final inputs/checksums and frozen review reports verified.')
print('Counts per mode:', plan['by_entity_partition_per_mode'])
print('External surface-review record IDs:', plan['externally_flagged_record_ids'])


In [ ]:
# Fresh public base only: no tiny M1 adapter or private checkpoint restoration.
run_env = os.environ.copy()
try:
    from kaggle_secrets import UserSecretsClient
    download_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    download_token = None
if download_token:
    run_env['HF_TOKEN'] = download_token
output_dir = Path(tempfile.mkdtemp(prefix='m0-qwen17-full-', dir=remote_root))
print('Audit outputs:', output_dir, flush=True)
try:
    subprocess.run([sys.executable, '-u', '-m', 'src.evaluation.full_m0', '--config', 'configs/evaluation/m0_qwen17_full_v1.json', '--output-dir', str(output_dir), '--mode', 'audit'], env=run_env, check=True)
finally:
    run_env.pop('HF_TOKEN', None)


In [ ]:
summary = json.loads((output_dir / 'summary.json').read_text())
assert summary['execution_complete'] and summary['training_updates'] == 0
assert not summary['adapters_loaded'] and not summary['weights_updated'] and not summary['dataset_accepted']
for mode, data in summary['modes'].items():
    score = data['answer_scoring']
    print(mode, '| answer matches:', score['answer_matches'], '/1260', '| format:', score['format_violations'], '| caps:', score['token_limit_hits'], '| missing boundaries:', data['missing_thinking_boundaries'], '| trace mentions:', data['reasoning_answer_mentions'], '| priority answer-review rows:', data['potential_answer_review_rows'])
print('Review files: potential_answers_for_review.json, protocol_issues_for_review.json, external_collision_rows_for_review.json')
print('All raw rows remain in the prediction files. Review findings before dataset acceptance; full M1 training is not authorized by this execution flag.')


In [ ]:
import zipfile

archive = Path(str(output_dir) + '-evidence.zip')
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(output_dir.iterdir()):
        if file.is_file() and file.suffix in {'.json', '.jsonl', '.txt'}:
            bundle.write(file, arcname=file.name)
print('Download from Kaggle Output:', archive)
